In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 50)

In [ ]:
input_path = "../data/raw/classified_segmented_data.parquet"
output_path = "../data/processed/cleaned_classified_segmented_data.parquet"

In [ ]:
df = pd.read_parquet(input_path, engine="pyarrow")
df.head()

# Remove missing entries

In [ ]:
pd.set_option('display.max_colwidth', None)

before_size = df.shape[0]
df = df.dropna(how='any') # Remove entire row if any value is missing
df.reset_index(drop=True, inplace=True)

print(f"Removed {before_size - df.shape[0]} rows with missing values.")

# `is_economic` basic statistics

In [ ]:
is_economic_df = df[df['is_economic'] == True].reset_index(drop=True)
non_economic_df = df[df['is_economic'] == False].reset_index(drop=True)

In [ ]:
is_economic_df[['title', 'url']].head(5)

In [ ]:
non_economic_df[['title', 'url']].head(5)

In [ ]:
remove_count = df.shape[0] - is_economic_df.shape[0]
print(f"Removed {remove_count}/{df.shape[0]} non-economic rows. ({remove_count / df.shape[0]:.2%})")

# Join with `time` in original data

In [ ]:
original_df = pd.read_csv("../data/processed/all_articles.csv")

final_df = pd.merge(is_economic_df, original_df[['time', 'url']], on='url', how='left')
final_df = final_df[['url', 'title', 'time', 'chunks']]
final_df.head()

# Save

In [ ]:
final_df.to_parquet(output_path, engine="pyarrow", index=False)
total_chunks = sum(final_df['chunks'].apply(len))
print(f"Saved {final_df.shape[0]:,} articles to {output_path}.")
print(f"Total chunks: {total_chunks:,}.")